# Patent Topic Modeling with LDA

In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import LatentDirichletAllocation
from pathlib import Path
import re

# specify which corpus version to load (used to build the file path below)
version_prefix = "v4"

## Data Loading and Preprocessing

In [3]:
# load data for the chosen prefix
base_data = Path("../../data/claims_added")
data_path = base_data / f"{version_prefix}_processed.csv"

df = pd.read_csv(data_path)

print(f"Dataset loaded: {df.shape[0]} patents")
df['text'] = df['claims'] # downstream code uses this name

# Quick tensor check
tensor_count = df['text'].str.contains('tensor', case=False, na=False).sum()
print(f"Patents containing 'tensor': {tensor_count}")

# load custom stopwords from file
stopfile = Path("custom_stopwords.txt")
if not stopfile.exists():
    raise FileNotFoundError(f"custom stopwords file not found: {stopfile}")
with open(stopfile, "r") as f:
    custom_stopwords = [line.strip() for line in f if line.strip()]

# text cleaning utility
def clean_text(s: str) -> str:
    s = s.lower()
    s = re.sub(r"[^a-z0-9_\s\-]", " ", s)         # keep alnum/_/- as token-friendly
    s = re.sub(r"\b\d{1,2}\b", " ", s)             # remove standalone 1-2 digit numbers
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_clean"] = df["text"].map(clean_text)

# Convert to list for processing
docs = df["text_clean"].astype(str).tolist()

# Combine English stopwords with our custom patent stopwords (only single words)
stop_words = set(ENGLISH_STOP_WORDS)
for w in custom_stopwords:
    for part in w.split():
        stop_words.add(part.lower())

vectorizer = CountVectorizer(
    stop_words=list(stop_words),
    max_df=0.4,
    min_df=10,
    ngram_range=(1, 2),
    lowercase=True,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9_]{1,}\b",
    binary=False
)

# Fit and transform the documents
X = vectorizer.fit_transform(docs)

print(f"Vocabulary size: {X.shape[1]} terms")
print(f"Document-term matrix shape: {X.shape}")
print(f"Sparsity: {(1 - X.nnz / (X.shape[0] * X.shape[1])):.1%}")

Dataset loaded: 6031 patents
Patents containing 'tensor': 195
Vocabulary size: 16118 terms
Document-term matrix shape: (6031, 16118)
Sparsity: 99.2%


## LDA Model Fitting

In [5]:
n_topics = 20

# Fit LDA model
lda = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=42,
    learning_method="batch",       
    max_iter=100,                  
    doc_topic_prior=0.3,           
    topic_word_prior=0.01 
)         

print("Fitting LDA model...")
lda.fit(X)

# Transform documents to get topic distributions
doc_topic_dist = lda.transform(X)

print(f"LDA fitted with {n_topics} topics")
print("Avg max topic weight:", doc_topic_dist.max(axis=1).mean())
print(f"Final log-likelihood: {lda.score(X):.2f}")

Fitting LDA model...
LDA fitted with 20 topics
Document-topic distribution shape: (6031, 20)
Each row contains P(topic_i | document) for topics 0-19
Avg max topic weight: 0.4413771524695742
Final log-likelihood: -28479394.47


## View topics

In [6]:
feature_names = vectorizer.get_feature_names_out()
topic_summaries = {}
for topic_idx, topic in enumerate(lda.components_):
    top_word_indices = topic.argsort()[-15:][::-1]  # Sort descending
    top_words = [feature_names[i] for i in top_word_indices]
    topic_summaries[topic_idx] = top_words
    print(f"Topic {topic_idx:02d}: " + ", ".join([f"{w}" for w in top_words]))

Topic 00: signal, digital, frequency, clock, signals, output, input, domain, complex, phase, control, component, according, generate, analog
Topic 01: value, quantization, step, according, values, quantized, number, threshold, error, predetermined, range, coefficient, maximum, calculating, difference
Topic 02: set, computer readable, transitory, non transitory, machine, model, transitory computer, learning, cause, processors, readable medium, executed, readable storage, storage, computing
Topic 03: network, neural, neural network, layer, quantization, model, quantized, layers, training, network model, parameters, output, bit, parameter, precision
Topic 04: information, target, according, parameter, feature, quantization, point, quantization parameter, obtain, region, preset, attribute, points, obtaining, cloud
Topic 05: address, cache, thread, execution, access, read, write, threads, request, compute, shared, group, core, local, queue
Topic 06: array, elements, element, sub, matrix, sy

In [ ]:
from stability import compute_topic_stability

k_values = [16, 18, 20, 22, 24, 26, 28]
seeds = [0, 1, 2, 3, 4]

results_df = compute_topic_stability(k_values, seeds, X, method='lda')

print("\n===== SUMMARY =====")
print(results_df.sort_values("k"))